In [2]:
import torch
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstD, ScaleIntensityRanged, Resize, ToTensord
from monai.networks.nets import UNet  # or your custom model
import matplotlib.pyplot as plt
from preprocess import RGBToGray

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(  # adapt if using a different model
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2,
).to(device)

In [7]:
model.load_state_dict(torch.load("models/best_metric_model.pth", map_location=device))
model.eval()

RuntimeError: Error(s) in loading state_dict for UNet:
	Missing key(s) in state_dict: "model.1.submodule.1.submodule.1.submodule.0.conv.unit0.conv.weight", "model.1.submodule.1.submodule.1.submodule.0.conv.unit0.conv.bias", "model.1.submodule.1.submodule.1.submodule.0.conv.unit0.adn.A.weight", "model.1.submodule.1.submodule.1.submodule.0.conv.unit1.conv.weight", "model.1.submodule.1.submodule.1.submodule.0.conv.unit1.conv.bias", "model.1.submodule.1.submodule.1.submodule.0.conv.unit1.adn.A.weight", "model.1.submodule.1.submodule.1.submodule.0.residual.weight", "model.1.submodule.1.submodule.1.submodule.0.residual.bias", "model.1.submodule.1.submodule.1.submodule.1.submodule.conv.unit0.conv.weight", "model.1.submodule.1.submodule.1.submodule.1.submodule.conv.unit0.conv.bias", "model.1.submodule.1.submodule.1.submodule.1.submodule.conv.unit0.adn.A.weight", "model.1.submodule.1.submodule.1.submodule.1.submodule.conv.unit1.conv.weight", "model.1.submodule.1.submodule.1.submodule.1.submodul

In [ ]:
infer_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstD(keys=["image"]),
    RGBToGray(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=255, b_min=0.0, b_max=1.0, clip=True),
    Resize(spatial_size=(256, 256), mode="bilinear"),
    ToTensord(keys=["image"])
])

In [ ]:
img_path = "data/TestImages/1048.png"  # Replace with your test image path
test_data = {"image": img_path}
input_data = infer_transforms(test_data)
input_tensor = input_data["image"].unsqueeze(0).to(device)

In [ ]:
with torch.no_grad():
    output = model(input_tensor)
    output = torch.sigmoid(output) 

In [ ]:
pred_mask = output.squeeze().cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title("Input Image")
plt.imshow(input_data["image"].squeeze().cpu().numpy(), cmap="gray")

plt.subplot(1, 2, 2)
plt.title("Predicted Mask")
plt.imshow(pred_mask > 0.5, cmap="gray")  # threshold the output
plt.show()